<a href="https://colab.research.google.com/github/mariselvi242007/research-paper-explainer-assistant/blob/main/notebooks/research_paper_explainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Explainer Assistant

This project uses Generative AI, embeddings, vector databases, and RAG
to explain research papers and answer questions based on their content.

In [23]:

# ============================================================
# PAPERLENS - RESEARCH PAPER EXPLAINER ASSISTANT

# ============================================================

%pip -q install -U groq google-genai pypdf scikit-learn

import io
import os
import numpy as np

from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from groq import Groq
from google import genai
from google.colab import userdata, files


# ============================================================
# 1. CONFIGURATION
# ============================================================

GEMINI_MODEL = "gemini-3-flash-preview"

PREFERRED_GROQ_MODELS = [
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
]


# ============================================================
# 2. LOAD API KEYS
# ============================================================

def get_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None


# Read keys from Colab Secrets first, then environment variables.
os.environ["GROQ_API_KEY"] = (
    get_secret("GROQ_API_KEY")
    or os.environ.get("GROQ_API_KEY", "")
)

os.environ["GEMINI_API_KEY"] = (
    get_secret("GEMINI_API_KEY")
    or os.environ.get("GEMINI_API_KEY", "")
)

groq_key = os.environ.get("GROQ_API_KEY")
gemini_key = os.environ.get("GEMINI_API_KEY")

if not groq_key and not gemini_key:
    raise ValueError(
        "Please add GROQ_API_KEY and/or GEMINI_API_KEY "
        "in Google Colab Secrets."
    )


# ============================================================
# 3. INITIALIZE GROQ FIRST
# ============================================================

groq_client = None
groq_model = None

if groq_key:
    try:
        groq_client = Groq(api_key=groq_key)

        available_models = groq_client.models.list()
        available_ids = [model.id for model in available_models.data]

        for candidate in PREFERRED_GROQ_MODELS:
            if candidate in available_ids:
                groq_model = candidate
                break

        if groq_model is None:
            for model_id in available_ids:
                if any(
                    name in model_id.lower()
                    for name in ["llama", "gpt-oss", "qwen"]
                ):
                    groq_model = model_id
                    break

        if groq_model:
            print("Groq ready:", groq_model)
        else:
            print("No suitable Groq model found.")

    except Exception as e:
        groq_client = None
        print("Groq initialization failed:", str(e)[:200])


# ============================================================
# 4. INITIALIZE GEMINI BACKUP
# ============================================================

gemini_client = None

if gemini_key:
    try:
        gemini_client = genai.Client(api_key=gemini_key)
        print("Gemini backup ready:", GEMINI_MODEL)
    except Exception as e:
        print("Gemini initialization failed:", str(e)[:200])


# ============================================================
# 5. UPLOAD AND READ PDF USING io
# ============================================================

print("\nUpload your research paper PDF:")
uploaded = files.upload()

pdf_files = [
    name for name in uploaded
    if name.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("Please upload a PDF file.")

pdf_name = pdf_files[0]

# io.BytesIO reads the uploaded PDF directly from memory.
pdf_bytes = io.BytesIO(uploaded[pdf_name])
reader = PdfReader(pdf_bytes)

page_texts = []

for page_number, page in enumerate(reader.pages, start=1):
    try:
        text = page.extract_text() or ""
        if text.strip():
            page_texts.append(text)
    except Exception as e:
        print(f"Could not read page {page_number}:", str(e)[:100])

document_text = "\n".join(page_texts)

if not document_text.strip():
    raise ValueError(
        "No readable text found. Scanned PDFs may require OCR."
    )

print("\nPDF loaded:", pdf_name)
print("Total pages:", len(reader.pages))
print("Extracted characters:", len(document_text))


# ============================================================
# 6. SPLIT TEXT INTO CHUNKS
# ============================================================

def create_chunks(text, chunk_size=250, overlap=40):
    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

        if start + chunk_size >= len(words):
            break

    return chunks


chunks = create_chunks(document_text)

if not chunks:
    raise ValueError("Could not create text chunks.")

print("Chunks created:", len(chunks))


# ============================================================
# 7. BUILD TF-IDF RETRIEVER
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000
)

chunk_vectors = vectorizer.fit_transform(chunks)


def retrieve_relevant_chunks(question, top_k=4):
    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector,
        chunk_vectors
    ).flatten()

    best_indices = np.argsort(scores)[::-1][:top_k]

    relevant_chunks = [
        chunks[i]
        for i in best_indices
        if scores[i] > 0
    ]

    if not relevant_chunks:
        relevant_chunks = [
            chunks[i] for i in best_indices
        ]

    return relevant_chunks


# ============================================================
# 8. BUILD PROMPT
# ============================================================

def build_prompt(question, context_chunks):
    context = "\n\n".join(
        f"[Paper excerpt {i + 1}]\n{chunk}"
        for i, chunk in enumerate(context_chunks)
    )

    return f"""
You are PaperLens, a research paper explainer assistant.

Answer using the research paper excerpts below.

Instructions:
- Use simple, clear English.
- Explain technical terms.
- Use bullet points when useful.
- Do not invent unsupported information.
- If the answer is not present in the excerpts, say so clearly.

RESEARCH PAPER EXCERPTS:
{context}

QUESTION:
{question}

ANSWER:
"""


# ============================================================
# 9. ASK GROQ
# ============================================================

def ask_groq(prompt):
    if groq_client is None or groq_model is None:
        raise RuntimeError("Groq is unavailable.")

    response = groq_client.chat.completions.create(
        model=groq_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are PaperLens. Explain research papers clearly "
                    "using the supplied excerpts."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1200
    )

    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        raise RuntimeError("Groq returned an empty response.")

    return answer.strip()


# ============================================================
# 10. ASK GEMINI
# ============================================================

def ask_gemini(prompt):
    if gemini_client is None:
        raise RuntimeError("Gemini is unavailable.")

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    answer = getattr(response, "text", None)

    if not answer or not answer.strip():
        raise RuntimeError("Gemini returned an empty response.")

    return answer.strip()


# ============================================================
# 11. AUTOMATIC SELECTION: GROQ -> GEMINI
# ============================================================

def answer_question(question, top_k=4):
    question = question.strip()

    if not question:
        return "Please enter a question."

    relevant_chunks = retrieve_relevant_chunks(question, top_k)
    prompt = build_prompt(question, relevant_chunks)
    errors = []

    # First attempt: Groq
    if groq_client is not None and groq_model is not None:
        try:
            print("Trying Groq first...")
            answer = ask_groq(prompt)

            return (
                "Answer:\n\n"
                + answer
                + f"\n\n[Model used: Groq - {groq_model}]"
            )

        except Exception as e:
            errors.append("Groq: " + str(e)[:300])
            print("Groq failed. Switching to Gemini...")

    else:
        errors.append("Groq: unavailable.")
        print("Groq unavailable. Trying Gemini...")

    # Backup attempt: Gemini
    if gemini_client is not None:
        try:
            print("Trying Gemini...")
            answer = ask_gemini(prompt)

            return (
                "Answer:\n\n"
                + answer
                + f"\n\n[Model used: Gemini - {GEMINI_MODEL}]"
            )

        except Exception as e:
            errors.append("Gemini: " + str(e)[:300])

    return (
        "Sorry, no AI provider could answer your question.\n\n"
        + "\n".join(errors)
        + "\n\nCheck your API keys, model access, and API limits."
    )


# ============================================================
# 12. ASK QUESTIONS ABOUT YOUR PAPER
# ============================================================

print("\n" + "=" * 50)
print("PAPERLENS IS READY!")
print("=" * 50)
print("Priority: Groq first, Gemini backup.")
print("Type 'exit' to stop.\n")

while True:
    question = input("Ask a question about your paper: ").strip()

    if question.lower() in ["exit", "quit", "stop"]:
        print("Thank you for using PaperLens!")
        break

    if not question:
        print("Please enter a question.\n")
        continue

    result = answer_question(question)

    print("\n" + result)
    print("\n" + "-" * 50 + "\n")

Groq ready: openai/gpt-oss-20b
Gemini backup ready: gemini-3-flash-preview

Upload your research paper PDF:


Saving artificial intelligence.pdf to artificial intelligence (1).pdf

PDF loaded: artificial intelligence (1).pdf
Total pages: 19
Extracted characters: 94931
Chunks created: 67

PAPERLENS IS READY!
Priority: Groq first, Gemini backup.
Type 'exit' to stop.

Ask a question about your paper: methodology
Trying Groq first...

Answer:

**Methodology**  
In the context of the excerpts, *methodology* refers to the set of steps, rules, or approaches that guide how an AI system is built, trained, and evaluated. It is the “recipe” that tells engineers and researchers what to do and why.

---

### What the excerpts say about methodology

| Area | What the excerpt says | What it means in plain English |
|------|-----------------------|--------------------------------|
| **Expert systems** | “The key to designing an expert system lies in utilizing expertise rather than methodology.” | When people built early AI that mimicked human experts, they focused more on copying human knowledge (rules, facts